# ConectaSucre · Generar embeddings vectoriales reales
Este cuaderno usa un modelo multilingüe de 384 dimensiones y **no requiere contraseña de MongoDB**. Al terminar descarga un archivo JavaScript para ejecutar en MongoDB Compass (Mongosh).
Los datos son incidentes sintéticos del proyecto. Comprueba que tus cinco textos no hayan cambiado.


In [ ]:
%pip -q install sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer
import json
from google.colab import files

# Modelo real de embeddings multilingües, 384 dimensiones.
modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

incidentes = [
    ("NOV-001", "Pinchazo de llanta durante el recorrido."),
    ("NOV-002", "Bloqueo temporal de la via por manifestacion."),
    ("NOV-003", "Cliente devuelve dos productos por empaque mojado."),
    ("NOV-004", "Sobrecalentamiento del motor durante el transporte."),
    ("NOV-005", "El mal estado de la via rural retraso la entrega.")
]
consulta = "El camion se detuvo porque fallo el sistema de refrigeracion del motor"
textos = [texto for _, texto in incidentes] + [consulta]
vectores = modelo.encode(textos, normalize_embeddings=True)
assert vectores.shape == (6, 384), f"Dimensiones inesperadas: {vectores.shape}"

lineas = [
    '// CONECTASUCRE - embeddings reales de 384 dimensiones',
    '// Ejecutar en mongosh conectado a Atlas. No borra documentos.',
    'var csVectorDB = db.getSiblingDB("conectasucre_nosql");',
    'var csVectorColeccion = csVectorDB.getCollection("novedades_logisticas");'
]
for (codigo, descripcion), vector in zip(incidentes, vectores[:-1]):
    array = json.dumps([round(float(x), 8) for x in vector], separators=(',', ':'))
    lineas.append('csVectorColeccion.updateOne(' + json.dumps({'codigo_novedad':codigo}) + ', {$set: {embedding: ' + array + ', modelo_embedding: "paraphrase-multilingual-MiniLM-L12-v2"}});')
lineas += [
    'print("Documentos con vectores de 384 dimensiones:");',
    'print(csVectorColeccion.countDocuments({"embedding.383": {$exists: true}}));',
    'print("Verifica que el índice idx_incidentes_semanticos esté READY antes de buscar.");',
]
consulta_vector = json.dumps([round(float(x), 8) for x in vectores[-1]], separators=(',', ':'))
lineas += [
    '// Consulta semántica: ejecutar después de que el índice esté READY',
    'var csQueryVector = ' + consulta_vector + ';',
    'printjson(csVectorColeccion.aggregate([{$vectorSearch: {index:"idx_incidentes_semanticos",path:"embedding",queryVector:csQueryVector,numCandidates:100,limit:5}},{$project:{_id:0,codigo_novedad:1,descripcion:1,puntaje:{$meta:"vectorSearchScore"}}}]).toArray());'
]
nombre = '12_CARGAR_Y_BUSCAR_VECTORES.js'
with open(nombre, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lineas) + '\n')
print(f"Archivo generado: {nombre}")
print("Vectores: 5 incidentes + 1 búsqueda, cada uno de 384 dimensiones.")
files.download(nombre)
